# Session 10 — Land-cover classification in Earth Engine
*Séance 10 — Classification de l'occupation du sol dans Earth Engine*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/10_classification_occupation_sol.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

| Training track · Parcours de formation | This session · Cette séance |
|---|---|
| Geomatics / environment · Géomatique / environnement (UdeS géomatique appliquée; Bishop's Environmental Studies & Geography) | ● full session / séance complète |
| Computer science · Informatique (UdeS informatique; Bishop's Computer Science) | ● full session / séance complète |
| College / continuing education · Cégep / formation continue | ● full session / séance complète |

**By the end of this session you will be able to:**

1. Build a cloud-free Sentinel-2 composite and add spectral indices as features.
2. Sample training data from an existing land-cover map (ESA WorldCover 2020).
3. Train and compare random forest, CART and SVM classifiers.
4. Apply the model to another year (2021) and tune the number of trees.

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

### Connect to Google Earth Engine
Since 13 November 2024, Earth Engine requires a **Google Cloud project** registered for Earth Engine (free for noncommercial use).
One-time setup: go to <https://code.earthengine.google.com/register>, choose *Unpaid usage → Academia & Research*, and create a project
(e.g. `ee-yourname`). Then write its ID below.

In [ ]:
import ee, geemap

EE_PROJECT = "ee-yourname"   # <-- replace with YOUR Cloud project ID

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()          # opens a sign-in link the first time
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready:", ee.String("Hello from Earth Engine").getInfo())

## Part A — Lecture

**Workflow.** (1) Image composite → (2) features (bands + indices) → (3) labelled samples → (4) train/test split →
(5) train classifiers → (6) evaluate → (7) predict a new year.

In 2022 we sampled labels from the South African National Land-Cover (SANLC) 2020. Here we use **ESA WorldCover**
(10 m, global, open: 2020 = v100, 2021 = v200), so the notebook works anywhere — including Quebec. The SANLC 2018
remains available in the community catalog (`projects/sat-io/open-datasets/landcover/SA_NLC_2018`) for South African
projects.

In [ ]:
import geopandas as gpd

WARD_FILES = {"ZA1011001": DATA / "SA_LEVEL4" / "ADM4_PCODE_ZA1011001.shp",   # Western Cape
              "ZA6372040": DATA / "SA_LEVEL4" / "ADM4_PCODE_ZA6372040.shp"}   # North West


def ward_fc(code):
    """Upload a ward polygon from the course repository to Earth Engine as a FeatureCollection."""
    return geemap.geopandas_to_ee(gpd.read_file(WARD_FILES[code])[["ADM4_PCODE", "geometry"]])


ward = ward_fc("ZA1011001")
aoi = ward.geometry()
print("Ward area (km²):", ward.geometry().area(maxError=1).divide(1e6).getInfo())

### 1. Cloud-free composite with Cloud Score+

In [ ]:
S2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
CSP = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]


def composite(year):
    """Median Sentinel-2 surface reflectance for one year, clouds masked with Cloud Score+ (cs_cdf >= 0.6)."""
    col = (S2.filterBounds(aoi).filterDate(f"{year}-01-01", f"{year + 1}-01-01")
             .linkCollection(CSP, ["cs_cdf"])
             .map(lambda img: img.updateMask(img.select("cs_cdf").gte(0.6))))
    img = col.select(BANDS).median().divide(10000).clip(aoi)
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = img.normalizedDifference(["B3", "B8"]).rename("NDWI")
    ndbi = img.normalizedDifference(["B11", "B8"]).rename("NDBI")
    return img.addBands([ndvi, ndwi, ndbi])


img2020, img2021 = composite(2020), composite(2021)
FEATURES = img2020.bandNames().getInfo()
Map = geemap.Map()
Map.centerObject(aoi, 11)
Map.addLayer(img2020, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, "S2 2020 true colour")
Map

### 2. Labels from ESA WorldCover 2020

In [ ]:
WC_CODES = [10, 20, 30, 40, 50, 60, 80, 90]          # tree, shrub, grass, crop, built-up, bare, water, wetland
WC_NAMES = ["Tree cover", "Shrubland", "Grassland", "Cropland", "Built-up", "Bare/sparse", "Water", "Wetland"]
WC_COLORS = ["006400", "ffbb22", "ffff4c", "f096ff", "fa0000", "b4b4b4", "0064c8", "0096a0"]


def worldcover(version):
    wc = ee.ImageCollection(f"ESA/WorldCover/{version}").first().clip(aoi)
    return wc.remap(WC_CODES, list(range(len(WC_CODES)))).rename("label")


labels2020 = worldcover("v100")
Map.addLayer(labels2020, {"min": 0, "max": 7, "palette": WC_COLORS}, "WorldCover 2020")
Map.add_legend(legend_keys=WC_NAMES, legend_colors=["#" + c for c in WC_COLORS])

### 3. Stratified sampling and split
We draw up to 300 points per class so rare classes (water, built-up) are represented, then split 70/30 at random.

In [ ]:
samples = (img2020.addBands(labels2020)
           .stratifiedSample(numPoints=300, classBand="label", region=aoi, scale=10, seed=42, geometries=True)
           .randomColumn("rnd", 42))
training = samples.filter(ee.Filter.lt("rnd", 0.7))
testing = samples.filter(ee.Filter.gte("rnd", 0.7))
print("Training:", training.size().getInfo(), "| Testing:", testing.size().getInfo())
print(samples.aggregate_histogram("label").getInfo())

### 4–6. Train, compare and evaluate

In [ ]:
import pandas as pd

classifiers = {"Random forest (100 trees)": ee.Classifier.smileRandomForest(100, seed=42),
               "CART": ee.Classifier.smileCart(),
               "SVM (RBF)": ee.Classifier.libsvm(kernelType="RBF", gamma=0.5, cost=10)}
trained, scores = {}, {}
for name, clf in classifiers.items():
    trained[name] = clf.train(features=training, classProperty="label", inputProperties=FEATURES)
    cm = testing.classify(trained[name]).errorMatrix("label", "classification")
    scores[name] = {"overall accuracy": cm.accuracy().getInfo(), "kappa": cm.kappa().getInfo()}
pd.DataFrame(scores).T.round(3)

## Part B — Lab

**Task 1 — Best model and confusion matrix.** Store the name of the model with the highest overall accuracy in `best`,
then put its test confusion matrix in a DataFrame `cm_df` (rows = reference, columns = predicted, labelled with
`WC_NAMES`).

In [ ]:
# YOUR CODE HERE
best = None
cm_df = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(best in scores)
check(cm_df.shape == (8, 8))

**Task 2 — Tune the forest.** Train random forests with 10, 25, 50, 100 and 200 trees and store the test accuracy of each
in a dict `tuning` (key = number of trees). Plot accuracy against the number of trees.

In [ ]:
# YOUR CODE HERE
tuning = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(sorted(tuning) == [10, 25, 50, 100, 200])

**Task 3 — Map 2021.** Classify `img2021` with the random forest (100 trees) into `map2021`, add it to the map, and compute
`agreement` = the share of pixels where `map2021` equals WorldCover 2021 (`worldcover("v200")`), at 30 m scale.

In [ ]:
# YOUR CODE HERE
map2021 = None
agreement = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(0 < agreement <= 1)

**Task 4 — Which features matter?** Use `rf.explain()` to get the variable importance and store it in a sorted pandas Series
`importance`.

In [ ]:
# YOUR CODE HERE
importance = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(len(importance) == len(FEATURES))

**Export the final map** (runs in the background; check the *Tasks* tab or `task.status()`):
```python
task = ee.batch.Export.image.toDrive(image=map2021.toByte(), description="landcover_2021", region=aoi, scale=10, maxPixels=1e9)
task.start()
```

**❓ Question —** Our 'truth' is itself a map (WorldCover) with its own errors. What does an accuracy of 90% against it really mean? Session 11 answers this.

*Your answer here.*

## Summary

- Composite → features → stratified samples → split → train → evaluate → predict.
- Random forests are a strong, fast baseline; compare with at least one other model.
- Labels sampled from an existing map inherit its errors: validate with independent reference data.

**To go further:**

- Zanaga, D. et al. (2022). ESA WorldCover 10 m 2021 v200. doi:10.5281/zenodo.7254221
- Pasquarella, V. J. et al. (2023). Comprehensive quality assessment of optical satellite imagery using weakly supervised video learning (Cloud Score+). CVPR Workshops.

---
*Adapted from the 2022 ML4EO bootcamp notebooks 'Introduction to machine learning for EO' and 'Generate the SANLC 2021' (N. C. Fouotsa Manfouo).*